In [2]:
import requests


url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2019/2019-10-08/ipf_lifts.csv'

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open('ipf_lifts.csv', 'wb') as file:
        for chunk in r.iter_content(chunk_size=8192):
            file.write(chunk)

In [3]:
import pandas as pd
from IPython.display import HTML, display


display(HTML("<style>.dataframe { font-size: 18px !important; }</style>"))

import seaborn as sns
# sns.set_theme(rc={'figure.figsize': (12, 6)})


df = pd.read_csv('ipf_lifts.csv')
df.info()
df


<class 'pandas.DataFrame'>
RangeIndex: 41152 entries, 0 to 41151
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   name              41152 non-null  str    
 1   sex               41152 non-null  str    
 2   event             41152 non-null  str    
 3   equipment         41152 non-null  str    
 4   age               38246 non-null  float64
 5   age_class         38268 non-null  str    
 6   division          40525 non-null  str    
 7   bodyweight_kg     40965 non-null  float64
 8   weight_class_kg   41151 non-null  str    
 9   best3squat_kg     27454 non-null  float64
 10  best3bench_kg     38690 non-null  float64
 11  best3deadlift_kg  27124 non-null  float64
 12  place             41152 non-null  str    
 13  date              41152 non-null  str    
 14  federation        41152 non-null  str    
 15  meet_name         41152 non-null  str    
dtypes: float64(5), str(11)
memory usage: 5.0 MB


,name,sex,event,equipment,age,age_class,division,bodyweight_kg,weight_class_kg,best3squat_kg,best3bench_kg,best3deadlift_kg,place,date,federation,meet_name
0,Hiroyuki Isagawa,M,SBD,Single-ply,NaN,NaN,NaN,67.5,67.5,205.0,140.0,225.0,1,1985-08-03,IPF,World Games
1,David Mannering,M,SBD,Single-ply,24.0,24-34,NaN,67.5,67.5,225.0,132.5,235.0,2,1985-08-03,IPF,World Games
2,Eddy Pengelly,M,SBD,Single-ply,35.5,35-39,NaN,67.5,67.5,245.0,157.5,270.0,3,1985-08-03,IPF,World Games
3,Nanda Talambanua,M,SBD,Single-ply,19.5,20-23,NaN,67.5,67.5,195.0,110.0,240.0,4,1985-08-03,IPF,World Games
4,Göran Henrysson,M,SBD,Single-ply,NaN,NaN,NaN,67.5,67.5,240.0,140.0,215.0,5,1985-08-03,IPF,World Games
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41147,Chien-Hsiung Chao,M,B,Single-ply,NaN,NaN,Open,126.5,125+,NaN,202.5,NaN,12,1995-06-25,IPF,World Bench Press Championships
41148,Oleg Gordynetz,M,B,Single-ply,NaN,NaN,Open,137.5,125+,NaN,202.5,NaN,13,1995-06-25,IPF,World Bench Press Championships
41149,Clive Lambert,M,B,Single-ply,31.5,24-34,Open,142.2,125+,NaN,202.5,NaN,14,1995-06-25,IPF,World Bench Press Championships
41150,Peter Brath,M,B,Single-ply,21.5,20-23,Open,125.5,125+,NaN,180.0,NaN,15,1995-06-25,IPF,World Bench Press Championships


In [4]:
exercise_to_max = ['best3bench_kg', 'best3squat_kg', 'best3deadlift_kg']

records_df = df.groupby(by=['sex', 'division'])[exercise_to_max].max()

records_df

best3bench_kg  best3squat_kg  best3deadlift_kg
sex division                                                   
F   Heavy                192.5          305.0             248.5
    Juniors              190.5          282.5             252.5
    Light                150.0          220.0             202.5
    Masters 1            197.5          245.0             248.0
    Masters 2            172.5          255.0             227.5
    Masters 3            140.0          210.0             190.0
    Masters 4            115.0          135.0             142.5
    Middle               170.0          247.5             235.0
    Open                 235.0          322.5             270.5
    Sub-Juniors          170.0          265.0             238.0
    Super                190.0          267.5             245.0
    SuperHeavy           205.0          310.5             247.5
M   Heavy                330.0          432.0             382.5
    Juniors              375.0          450.0             377.5
    Light                217.5          325.5             315.0
    Masters 1            345.0          405.0             395.0
    Masters 2            310.5          365.0             335.0
    Masters 3            260.0          300.0             305.0
    Masters 4            220.0          235.0             260.5
    Middle               268.5          370.0             345.0
    Open                 415.0          490.0             407.5
    Sub-Juniors          310.0          370.0             335.0
    Super                360.0          465.0             387.5
    SuperHeavy           405.0          475.0             420.0

In [12]:

df_winners = df[df['place'].astype(str) == '1']


count_of_wins = df_winners.groupby(by=['name', 'division']).size().reset_index(name='win_count')
count_of_wins


,name,division,win_count
0,A Ernandos-Ortega,Sub-Juniors,1
1,A. Raface,Masters 1,1
2,Aarre Käpylä,Juniors,1
3,Ab Wolders,Masters 3,2
4,Ab Wolders,Open,1
...,...,...,...
3876,Øyvind Bjørnsen,Juniors,1
3877,Þórunn Brynja Jónasdóttir,Open,1
3878,Štefan Koľšovský,Masters 1,5
3879,Štefan Koľšovský,Masters 2,2


In [30]:
merged_df = df.merge(count_of_wins, how='left', on=['name', 'division'])

merged_df['max_wins'] = merged_df.groupby(by=['sex', 'division'])['win_count'].transform('max')

best_lifters = merged_df[merged_df['win_count'] == merged_df['max_wins']]


final_df = best_lifters.drop_duplicates(subset=['name', 'sex', 'division'])

result_df = final_df[['division', 'sex', 'name', 'win_count']]
result_df





,division,sex,name,win_count
2199,Light,M,Sergey Fedosienko,2.0
2209,Middle,M,Jarosław Olech,3.0
2219,Heavy,M,Vadym Dovhanyuk,1.0
2222,Heavy,M,Sergii Bilyi,1.0
2229,Super,M,Andrey Konovalov #1,1.0
2239,Light,F,Natalia Salnikova,2.0
2249,Middle,F,Larysa Soloviova,2.0
2257,Heavy,F,Ana Rosa Castellain,2.0
2267,Super,F,Olena Kozlova,1.0
2314,Juniors,M,Kirill Krut,6.0
